# Graph Neural Network (GNN) for track reconstruction

## 1.- Introduction

The goals of this section are the following:

* To understand what a graph and a GNN are
* To learn to code graphs using pyTorch geometric
* Train a GNN and test the results
* Evaluate the performance of the reconstruction

Please download the libraries in order to start.

Attention: Set the runtime to GPU for this exercise.

In [ ]:
!rm -rf InstrumentationSchool/
!git clone https://github.com/parbol/InstrumentationSchool.git
import sys
sys.path.append('/content/InstrumentationSchool')

## 2.- Download the data

In the previous exercise you generated a few tracks using the Generic Tracker model. Since that was time consuming I have prepared already some files with different number of tracks. Please download.

In [ ]:
##### Getting the data
import urllib.request

urllib.request.urlretrieve('https://nextcloud.ifca.es/index.php/s/byo3piZP6Z2T7ZA/download', 'data50ParticlesTrain.parquet')
urllib.request.urlretrieve('https://nextcloud.ifca.es/index.php/s/EtD6xKQTbnKy9jx/download', 'data50ParticlesValidation.parquet')
urllib.request.urlretrieve('https://nextcloud.ifca.es/index.php/s/ngCypAKiw2oLWkR/download', 'data100ParticlesTrain.parquet')
urllib.request.urlretrieve('https://nextcloud.ifca.es/index.php/s/6sYrEEWXfpF9dsG/download', 'data100ParticlesValidation.parquet')
urllib.request.urlretrieve('https://nextcloud.ifca.es/index.php/s/Ad4Aa4SqiexBQjz/download', 'tracker.json')


print("Data downloaded successfully")

## 3.- Imports needed by the program

This program will use most of the modules that we used before including the GenericTracker for plotting purposes. The new module in this block is the pyTorch library to do GNNs called pyTorch geometric.



In [ ]:
! pip install torch_geometric
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch_geometric.transforms as T
from torch_geometric.data import HeteroData
import torch.nn.functional as F
from GNNForTracking.src.GNNModel import GNNModel
from GNNForTracking.src.DataBuilder import DataBuilder
from torch_geometric.nn import to_hetero
from GenericTrackerSimulator.src.Geometry.FullTracker import FullTracker
from GenericTrackerSimulator.src.Geometry.Tracker import Tracker
from GenericTrackerSimulator.src.Geometry.BarrelLayer import BarrelLayer
from GenericTrackerSimulator.src.Geometry.EndcapDisk import EndcapDisk
from GenericTrackerSimulator.src.Geometry.GeometryBuilder import GeometryBuilder
from GenericTrackerSimulator.src.Geometry.GeometryTools import GeometryTools
from GenericTrackerSimulator.src.Generation.genParticle import genParticle

## 4.- Some questions before you continue

Q. What is a graph?

A.

Q. What is a node and what are its features?

A.

Q. What is an edge and what are its weights or labels?

A.

Q. What is an homogeneous graph?

A.

Q. What is an heterogeneous graph?

A.

Q. What is a directed graph?

A.

## 5.- Read the data and adapt it to a GNN

The hit information is stored in the parquet files, however in order to create a graph we need to give structure to this data. In particular, each node will have 3 features (x, y, z) and in addition we need to create the edges. In order to simplify the complexity of the graph, we will only connect hits from one layer to the next, and only if they are in a cone of less than 45 degrees, which for high pt particles is well justified.

Please go to the DataBuilder class and understand how exactly this is done.

In [ ]:
# Select the device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Prepare and get the data
builderTrain = DataBuilder('data50ParticlesTrain.parquet', 45.0)
train_data = builderTrain.build()
builderVal = DataBuilder('data50ParticlesTrain.parquet', 45.0)
val_data = builderVal.build()

## 6. A question before you continue

Q. Suppose a graph with 4 nodes, the first is connected to the second and the third. The second is connected to the forth. The third is connected to all the others. The fourth is connected to the first one. How would be the tensor representing the edges of this graph in pyTorch Geometric?

A.

## 7.- Visualize the problem

Now we will visualize what the problem is. In this part you have to:

* Set up the tracker model and plot the tracker
* Read the files with the hits
* Make a function to plot the hits on top of the tracker

In [ ]:
def drawPoints(ax1, ax2, ax3, ax4, data):

    points = data.x_dict['source'].cpu().detach().numpy()
    # Modify accordingly


In [ ]:
# Setup the tracker
gBuilder = GeometryBuilder()
gTools = GeometryTools(gBuilder.ftr)
gTools.importGeometry('tracker.json')


In [ ]:
fig = plt.figure(figsize = (8, 8), layout="constrained")
gs0 = fig.add_gridspec(2, 1, height_ratios=[2,1])
ax1 = fig.add_subplot(gs0[0], projection = '3d')
gs1 = gs0[1].subgridspec(1,3)
ax2 = fig.add_subplot(gs1[0])
ax3 = fig.add_subplot(gs1[1])
ax4 = fig.add_subplot(gs1[2])
ax1.xaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
ax1.yaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
ax1.zaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
ax1.set_xlabel('x [cm]')
ax1.set_ylabel('z [cm]')
ax1.set_zlabel('y [cm]')
ax2.set_xlabel('x [cm]')
ax2.set_ylabel('y [cm]')
ax3.set_xlabel('z [cm]')
ax3.set_ylabel('y [cm]')
ax4.set_xlabel('z [cm]')
ax4.set_ylabel('x [cm]')

gBuilder.ftr.drawBarrel(ax1, ax2, ax3, ax4, t='b--')
drawPoints(ax1, ax2, ax3, ax4, train_data)


## 8.- Create the model

In this section we will setup the model and the optimizer. Please have a look at the model itself and try to understand all the elements.

In [ ]:
# Create model and optimizer
model = GNNModel(train_data.metadata(), hidden_channels=32).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)


## 9.- Perform the training

In this section, we will be performing the training of the model. Please identify clearly what is the loss function utilized and what is exactly being optimized.  


In [ ]:
# Actual training
train_data = train_data.to(device)
val_data = val_data.to(device)


theTrainLoss = 0
theValLoss = 0
for epoch in range(1, 3000):
  model.train()
  optimizer.zero_grad()
  pred = model(train_data.x_dict, train_data.edge_index_dict,
               train_data['source', 'target'].edge_index)
  target = train_data['source', 'target'].edge_label
  loss = F.mse_loss(pred, target)
  loss.backward()
  optimizer.step()
  theTrainLoss = loss.item()
  with torch.no_grad():
    model.eval()
    pred = model(val_data.x_dict, val_data.edge_index_dict,
                 val_data['source', 'target'].edge_index)
    pred = pred.clamp(min=0, max=1)
    target = val_data['source', 'target'].edge_label.float()
    theValLoss = F.mse_loss(pred, target).sqrt()
  print(f'Epoch: {epoch:03d}, Loss: {theTrainLoss:.4f}, Val: {theValLoss:.4f}')


## 10. Run the validation

In this block, we will run the full prediction over the full testing dataset. We will build a DataFrame in order to handle the different information. The key point is that if the prediction is higher than 0.5 then the edge is considered active and if not inactive.

In [ ]:
test_data = val_data

with torch.no_grad():
  test_data = test_data.to(device)
  # Modify accordingly

  pred = pred.clamp(min=0, max=1)
  target = test_data['source', 'target'].edge_label.float()
  rmse = F.mse_loss(pred, target).sqrt()
  print(f'Test RMSE: {rmse:.4f}')

sour = test_data['source', 'target'].edge_index[0].cpu().numpy()
tar = test_data['source', 'target'].edge_index[1].cpu().numpy()
pred = pred.cpu().numpy()
target = target.cpu().numpy()
res=pd.DataFrame({'source': sour, 'target': tar, 'pred': pred, 'compare': target})

#Add a new column if pred is greater or equal than 0.5 then 1 else 0.5
res['weight'] = np.where(res['pred']>=0.5, 1., 0.)



## 11. Estimate the metrics

In this block you will calculate the overall accuracy (how many times a predicted edge is actually right), and the overall accuracy for negative and positive edges separately.

In [ ]:
#compare column rating_1 with target and if they are equal add up
#Modify accordingly

#Calculate the overal efficiency accuracy
accuracy = cont/len(res)
print('Accuracy:', accuracy)
print('Number of correct predictions:', cont)


#Calculate the overal efficiency for connected and not connected edges
connected_accuracy = 0.
nonconnected_accuracy = 0.

#Modify accordingly

connected_accuracy = n2/ncon
nonconnected_accuracy = n1/nncon

print(f'Accuracy in connected edges:     {n2}/{ncon} = {connected_accuracy}')
print(f'Accuracy in non connected edges: {n1}/{nncon} = {nonconnected_accuracy}')



## 12. Visualize the results

This section is very similar to section number 7, however now, in the function drawing the points you should also draw lines between the nodes actually connected by an edge. You can use the same tracker model as before.

In [ ]:
def drawPointsAndEdges(ax1, ax2, ax3, ax4, data, res):

    points = data.x_dict['source'].cpu().detach().numpy()
    edges = data['source', 'target'].edge_index.cpu().detach().numpy()

    #Modify accordingly


In [ ]:
fig = plt.figure(figsize = (8, 8), layout="constrained")
gs0 = fig.add_gridspec(2, 1, height_ratios=[2,1])
ax1 = fig.add_subplot(gs0[0], projection = '3d')
gs1 = gs0[1].subgridspec(1,3)
ax2 = fig.add_subplot(gs1[0])
ax3 = fig.add_subplot(gs1[1])
ax4 = fig.add_subplot(gs1[2])
ax1.xaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
ax1.yaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
ax1.zaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
ax1.set_xlabel('x [cm]')
ax1.set_ylabel('z [cm]')
ax1.set_zlabel('y [cm]')
ax2.set_xlabel('x [cm]')
ax2.set_ylabel('y [cm]')
ax3.set_xlabel('z [cm]')
ax3.set_ylabel('y [cm]')
ax4.set_xlabel('z [cm]')
ax4.set_ylabel('x [cm]')

gBuilder.ftr.drawBarrel(ax1, ax2, ax3, ax4, t='b--')

drawPointsAndEdges(ax1, ax2, ax3, ax4, test_data, res)